<a href="https://colab.research.google.com/github/vikrampal12345/LangGraph/blob/main/Parellel_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Qwen3-8B"

In [ ]:
import os
MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen3-8B"

print(os.listdir(MODEL_PATH)[:20])

['.cache', 'generation_config.json', 'LICENSE', 'README.md', '.gitattributes', 'model.safetensors.index.json', 'merges.txt', 'config.json', 'vocab.json', 'tokenizer_config.json', 'tokenizer.json', 'model-00005-of-00005.safetensors', 'model-00002-of-00005.safetensors', 'model-00001-of-00005.safetensors', 'model-00004-of-00005.safetensors']


In [ ]:
from pathlib import Path
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_PATH = Path("/content/drive/MyDrive/AI_Models/Qwen3-8B")

print("Model folder exists:", MODEL_PATH.exists())
print("Config exists:", (MODEL_PATH / "config.json").exists())

Model folder exists: False
Config exists: False


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "config.json" in files and "Qwen" in root:
        print("FOUND:", root)

In [ ]:
!find /content/drive/MyDrive -type f -name "config.json" 2>/dev/null


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

print(os.path.exists("/content/drive"))
print(os.listdir("/content/drive"))

True
['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [ ]:
print(os.listdir("/content/drive/MyDrive"))

['placement.csv', 'Colab Notebooks', '.ipynb_checkpoints', 'titanic_pipeline.pkl', 'Earthquake_pipeline.pkl', 'AI_Models']


In [ ]:
print(os.listdir("/content/drive/MyDrive/AI_Models"))

['Qwen3-8B']


In [ ]:
print(os.listdir("/content/drive/MyDrive/AI_Models/Qwen3-8B")[:20])

['.cache', 'generation_config.json', 'LICENSE', 'README.md', '.gitattributes', 'model.safetensors.index.json', 'merges.txt', 'config.json', 'vocab.json', 'tokenizer_config.json', 'tokenizer.json', 'model-00005-of-00005.safetensors', 'model-00002-of-00005.safetensors', 'model-00001-of-00005.safetensors', 'model-00004-of-00005.safetensors']


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen3-8B"

hf_hub_download(
    repo_id="Qwen/Qwen3-8B",
    filename="model-00003-of-00005.safetensors",
    local_dir=MODEL_PATH
)

print("Missing shard downloaded.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:762: UserWarning: Not enough free disk space to download the file. The expected file size is: 3959.60 MB. The target location /content/drive/MyDrive/AI_Models/Qwen3-8B/.cache/huggingface/download only has 3625.48 MB free disk space.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:762: UserWarning: Not enough free disk space to download the file. The expected file size is: 3959.60 MB. The target location /content/drive/MyDrive/AI_Models/Qwen3-8B only has 3625.48 MB free disk space.
  warnings.warn(


model-00003-of-00005.safetensors: reconstructing file:   0%|          |  0.00B / 3.96GB            

model-00003-of-00005.safetensors: downloading bytes:           |  0.00B            

Missing shard downloaded.


In [ ]:
import os

for i in range(1, 6):
    file = f"{MODEL_PATH}/model-{i:05d}-of-00005.safetensors"
    print(os.path.basename(file), "->", os.path.exists(file))

model-00001-of-00005.safetensors -> True
model-00002-of-00005.safetensors -> True
model-00003-of-00005.safetensors -> True
model-00004-of-00005.safetensors -> True
model-00005-of-00005.safetensors -> True


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen3-8B"

# 4-bit quantization for T4 16GB GPU
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
    local_files_only=True
)

print("Qwen3-8B loaded successfully!")
print("Device:", model.device)

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI assistant."
    },
    {
        "role": "user",
        "content": "Explain what an AI agent is in simple words."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

In [ ]:
!pip install -U -q "bitsandbytes>=0.46.1"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
    local_files_only=True
)

print("Qwen3-8B loaded!")
print("Device:", model.device)

In [ ]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

In [ ]:
import sys
import bitsandbytes as bnb

print("Python:", sys.version)
print("bitsandbytes:", bnb.__version__)
print("bitsandbytes path:", bnb.__file__)

!nvidia-smi

In [ ]:
import bitsandbytes as bnb

print("Version:", bnb.__version__)

try:
    print("CUDA available to bitsandbytes:", bnb.cuda.is_available())
except Exception as e:
    print("CUDA check error:", repr(e))

In [ ]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

try:
    from bitsandbytes.cextension import lib
    print("bitsandbytes native library loaded:", lib)
    print("SUCCESS: bitsandbytes CUDA backend loaded")
except Exception as e:
    print("FAILED:", type(e).__name__)
    print(e)

In [ ]:
import transformers
from transformers.utils import is_bitsandbytes_available

print("Transformers:", transformers.__version__)
print("bitsandbytes available to Transformers:", is_bitsandbytes_available())

In [ ]:
import transformers
from transformers.utils import import_utils

print("Transformers:", transformers.__version__)
print("Required bitsandbytes version:",
      import_utils.BITSANDBYTES_MIN_VERSION)

print("is_bitsandbytes_available:",
      import_utils.is_bitsandbytes_available())

print("Package check:")
print(import_utils._is_package_available(
    "bitsandbytes",
    return_version=True
))

In [ ]:
import_utils._is_package_available("bitsandbytes", return_version=True)

In [ ]:
import inspect
from transformers.utils import import_utils

print(inspect.getsource(import_utils.is_bitsandbytes_available))

In [ ]:
from transformers.utils import import_utils
from packaging import version

print("Detected version:",
      import_utils._is_package_available("bitsandbytes", return_version=True)[1])

print("Transformers minimum:",
      import_utils.BITSANDBYTES_MIN_VERSION)

print(
    "Version comparison:",
    version.parse("0.50.2") >= version.parse(
        import_utils.BITSANDBYTES_MIN_VERSION
    )
)

In [ ]:
@lru_cache
@_make_compile_constant

In [ ]:
from transformers.utils import import_utils

print("Before clearing:")
print(import_utils.is_bitsandbytes_available())

import_utils.is_bitsandbytes_available.cache_clear()

print("After clearing:")
print(import_utils.is_bitsandbytes_available())

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

print("Loading Qwen3-8B...")

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
    local_files_only=True
)

print("\n✅ Qwen3-8B loaded successfully!")
print("Device:", qwen_model.device)

In [ ]:
from typing import Optional, List
from langchain_core.language_models.llms import LLM

class LocalQwenLLM(LLM):

    @property
    def _llm_type(self) -> str:
        return "local-qwen"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None
    ) -> str:

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful AI assistant. "
                    "Do not show your reasoning or thinking process. "
                    "Give only the final answer."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(qwen_model.device)

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=500,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        return response


model = LocalQwenLLM()

print("✅ Local Qwen LangChain model created")

In [ ]:
response = model.invoke(
    "Explain what an AI agent is in simple words."
)

print(response)

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field

In [ ]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description='Detailed feedback for the essay')
    score : int = Field(description='Score out of 10', ge=0, le = 10)

In [ ]:
import json
from pydantic import BaseModel, Field

class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)


essay = """
Artificial intelligence is changing education by helping teachers
personalize learning and automate repetitive tasks.
"""

prompt = f"""
Evaluate the following essay.

Return ONLY valid JSON.
Do not use markdown.
Do not explain anything outside the JSON.

The JSON must have exactly these fields:
{{
    "feedback": "detailed feedback about the essay",
    "score": 0
}}

The score must be an integer from 0 to 10.

Essay:
{essay}
"""

response = model.invoke(prompt)

print(response)

In [ ]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description='Detailed feedback for the essay')
    score : int = Field(description='Score out of 10', ge=0, le = 10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = """The Peacock is the national bird of India. Its scientific name is
Pavo cristatus. It is known for its beautiful feathers and graceful dance.

The male peacock has colourful feathers with eye-shaped patterns.
During the rainy season, it spreads its feathers and performs a
beautiful dance. The female is called a peahen.

Peacocks are found in many parts of India. They eat grains, seeds,
fruits, insects and small animals. They are an important part of
India's natural heritage.

The peacock was declared the National Bird of India in 1963.
We should protect peacocks and their natural habitats for future
generations."""

In [ ]:
prompt = f"""Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"""
structured_model.invoke(prompt)

In [ ]:
from typing import Optional, List, Type
import json

from langchain_core.language_models.llms import LLM
from langchain_core.runnables import RunnableLambda


class LocalQwenLLM(LLM):

    @property
    def _llm_type(self) -> str:
        return "local-qwen"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None
    ) -> str:

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful AI assistant. "
                    "Do not show your reasoning or thinking process. "
                    "Give only the final answer."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(qwen_model.device)

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=500,
                temperature=0.2,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        return response

    def with_structured_output(self, schema, **kwargs):

        def structured_call(prompt):

            schema_json = json.dumps(
                schema.model_json_schema(),
                indent=2
            )

            structured_prompt = f"""
{prompt}

You MUST return ONLY valid JSON.

Follow this JSON schema:

{schema_json}

Do not use markdown.
Do not use ```json.
Do not add explanations outside the JSON.
"""

            response = self.invoke(structured_prompt)

            # Remove accidental markdown fences
            response = response.strip()

            if response.startswith("```"):
                response = response.replace("```json", "")
                response = response.replace("```", "")
                response = response.strip()

            data = json.loads(response)

            return schema.model_validate(data)

        return RunnableLambda(structured_call)


model = LocalQwenLLM()

print("✅ Local Qwen model ready")

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

result = structured_model.invoke(prompt)

print("Score:", result.score)
print("Feedback:", result.feedback)

In [ ]:
essay = """The Peacock is the national bird of India. Its scientific name is
Pavo cristatus. It is known for its beautiful feathers and graceful dance.

The male peacock has colourful feathers with eye-shaped patterns.
During the rainy season, it spreads its feathers and performs a
beautiful dance. The female is called a peahen.

Peacocks are found in many parts of India. They eat grains, seeds,
fruits, insects and small animals. They are an important part of
India's natural heritage.

The peacock was declared the National Bird of India in 1963.
We should protect peacocks and their natural habitats for future
generations."""

In [ ]:
prompt = f"""
Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10

{essay}
"""

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)
result = structured_model.invoke(prompt)
print(result)


In [ ]:
import operator
from typing import Annotated

In [ ]:
class UPSCState(TypedDict):

  essay: str
  language_feedback: str
  analysis_feedback: str
  clarity_feedback: str
  overall_feedback: str
  individual_scores: Annotated[list[int], operator.add]  # operator add used because all the score generated simulteanous so may be overwrite so this is used
  avg_score: float



In [ ]:
def evaluate_language(state: UPSCState):
  prompt = f"""
  Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10

  {state["essay"]}
  """
  output = structured_model.invoke(prompt)

  return {'language_feedback': output.feedback, 'individual_scores' : [output.score]}



In [ ]:
def evaluate_analysis(state: UPSCState):
  prompt = f"""
  Evaluate the depth of analysis quality of the following essay and provide a feedback and assign a score out of 10

  {state["essay"]}
  """
  output = structured_model.invoke(prompt)

  return {'analysis_feedback': output.feedback, 'individual_scores' : [output.score]}

In [ ]:
def evaluate_thought(state: UPSCState):
  prompt = f"""
  Evaluate the clarity of thought quality of the following essay and provide a feedback and assign a score out of 10

  {state["essay"]}
  """
  output = structured_model.invoke(prompt)

  return {'clarity_feedback': output.feedback, 'individual_scores' : [output.score]}

In [ ]:
def final_evaluation(state: UPSCState):

  # summary feedback
  prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]} '

  overall_feedback = model.invoke(prompt)

  # avg calculate
  avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

  return {'overall_feedback': overall_feedback, 'avg_score' : avg_score}


In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

In [ ]:
# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')

graph.add_edge('evaluate_analysis', 'final_evaluation')

graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

graph.compile()

In [ ]:
workflow = graph.compile()

In [ ]:
initial_state = {
    'essay' : essay
}
workflow.invoke(initial_state)